# Web Search & Content Extraction Tools — Demo

Quick demos of `google_search` and `fetch_page_as_markdown`.

In [1]:
import sys, json
sys.path.insert(0, '..')

from tools.web_search import GoogleSearchInput, GoogleSearchResult, google_search
from tools.web_content import FetchPageInput, PageMarkdownResult, fetch_page_as_markdown
from IPython.display import Markdown, display

## 1. Fetch example.com — Simplest possible test

In [2]:
args = FetchPageInput(url="https://example.com")
raw = fetch_page_as_markdown(args)
result = PageMarkdownResult.model_validate_json(raw)

print(f"Title: {result.title}")
print(f"Error: {result.error}")
print(f"Markdown length: {len(result.markdown)} chars")
print("---")
display(Markdown(result.markdown))

[2026-03-19 17:05:38] INFO: Fetched (200) <GET https://example.com/> (referer: https://www.google.com/)


Title: Example Domain
Error: None
Markdown length: 168 chars
---


# Example Domain

This domain is for use in documentation examples without needing permission. Avoid use in operations.

[Learn more](https://iana.org/domains/example)


## 2. Google Search — "Pikachu pokemon"

In [3]:
args = GoogleSearchInput(query="Pikachu pokemon", max_results=5)
raw = google_search(args)
result = GoogleSearchResult.model_validate_json(raw)

print(f"Query: {result.query}")
print(f"Error: {result.error}")
print(f"Results: {result.total_returned}\n")

for i, r in enumerate(result.results):
    print(f"[{i+1}] {r.title}")
    print(f"    {r.url}")
    if r.snippet:
        print(f"    {r.snippet[:120]}")
    print()

[2026-03-19 17:05:44] INFO: Fetched (200) <GET https://www.google.com/search?q=Pikachu+pokemon&num=5&hl=en&sei=1x68aeGKC-iF9u8PrOXeqAo> (referer: https://www.google.com/)


Query: Pikachu pokemon
Error: None
Results: 5

[1] Pikachu (Pokémon) - Bulbapedia
    https://bulbapedia.bulbagarden.net/wiki/Pikachu_(Pok%C3%A9mon)
    Bulbapedia https://bulbapedia.bulbagarden.net › wiki › Pikachu_(Po...

[2] Pikachu – PokéWiki
    https://www.pokewiki.de/Pikachu
    pokewiki.de https://www.pokewiki.de › Pikachu

[3] Pikachu Pokédex: stats, moves, evolution & locations
    https://pokemondb.net/pokedex/pikachu
    Pokemon Database https://pokemondb.net › pokedex › pikachu

[4] 025 — Pikachu im Pokédex
    https://www.bisafans.de/pokedex/025.php
    Bisafans.de https://www.bisafans.de › pokedex

[5] Pikachu | Pokédex
    https://www.pokemon.com/us/pokedex/pikachu
    Pokemon.com https://www.pokemon.com › pokedex › pikachu



## 3. Google Search — Site-restricted to Bulbapedia

In [4]:
args = GoogleSearchInput(
    query="Charizard",
    site_restrict="bulbapedia.bulbagarden.net",
    max_results=5,
)
raw = google_search(args)
result = GoogleSearchResult.model_validate_json(raw)

print(f"Query: {result.query}")
print(f"Results: {result.total_returned}\n")

for i, r in enumerate(result.results):
    print(f"[{i+1}] {r.title}")
    print(f"    {r.url}\n")

[2026-03-19 17:06:33] INFO: Fetched (200) <GET https://www.google.com/search?q=site%3Abulbapedia.bulbagarden.net+Charizard&num=5&hl=en&sei=Bx-8aeaBOfyVxc8Px4Pd6Q4> (referer: https://www.google.com/)


Query: site:bulbapedia.bulbagarden.net Charizard
Results: 4

[1] Charizard (Pokémon) - Bulbapedia
    https://bulbapedia.bulbagarden.net/wiki/Charizard_(Pok%C3%A9mon)

[2] Charizard (TCG) - Bulbapedia
    https://bulbapedia.bulbagarden.net/wiki/Charizard_(TCG)

[3] Ash's Charizard - Bulbapedia, the community-driven Pokémon ...
    https://bulbapedia.bulbagarden.net/wiki/Ash%27s_Charizard

[4] Charizard (Pokémon)/Generation III learnset - Bulbapedia
    https://bulbapedia.bulbagarden.net/wiki/Charizard_(Pok%C3%A9mon)/Generation_III_learnset



## 4. Fetch Bulbapedia — Pikachu page (stealth mode)

Bulbapedia is behind Cloudflare, so we use `use_stealth=True`.

In [6]:
args = FetchPageInput(
    url="https://bulbapedia.bulbagarden.net/wiki/Pikachu_(Pok%C3%A9mon)",
    css_selector="#mw-content-text",
    use_stealth=True,
)
raw = fetch_page_as_markdown(args)
result = PageMarkdownResult.model_validate_json(raw)

print(f"Title: {result.title}")
print(f"Error: {result.error}")
print(f"Markdown length: {len(result.markdown):,} chars")
print("---")
# Show first 2000 chars as rendered markdown
display(Markdown(result.markdown[:20000] + "\n\n*... (truncated) ...*"))

[2026-03-19 17:07:52] INFO: Fetched (200) <GET https://bulbapedia.bulbagarden.net/wiki/Pikachu_(Pok%C3%A9mon)> (referer: https://www.google.com/)


Title: Pikachu (Pokémon) - Bulbapedia, the community-driven Pokémon encyclopedia
Error: None
Markdown length: 484,497 chars
---




- For Pokémon GO information on this species, see [the game's section](#Pok%C3%A9mon_GO).
- [←](/wiki/Arbok_(Pok%C3%A9mon) "Arbok (Pokémon)") [/wiki/Arbok_(Pok%C3%A9mon)](/wiki/Arbok_(Pok%C3%A9mon) "Arbok (Pokémon)") [#0024: Arbok](/wiki/Arbok_(Pok%C3%A9mon) "Arbok (Pokémon)") | [Pokémon](/wiki/List_of_Pok%C3%A9mon_by_National_Pok%C3%A9dex_number "List of Pokémon by National Pokédex number") |
| --- | - [#0026: Raichu](/wiki/Raichu_(Pok%C3%A9mon) "Raichu (Pokémon)") [/wiki/Raichu_(Pok%C3%A9mon)](/wiki/Raichu_(Pok%C3%A9mon) "Raichu (Pokémon)") [→](/wiki/Raichu_(Pok%C3%A9mon) "Raichu (Pokémon)")
- This article is about the species. For a specific instance of this species, see [Pikachu (disambiguation)](/wiki/Pikachu_(disambiguation) "Pikachu (disambiguation)").

| | | **Pikachu**   [Mouse Pokémon](/wiki/Pok%C3%A9mon_category "Pokémon category") | **ピカチュウ**   *Pikachu* | | --- | --- | | [#0025](/wiki/List_of_Pok%C3%A9mon_by_National_Pok%C3%A9dex_number "List of Pokémon by National Pokédex number") | | --- | --- | | - [Pikachu](/wiki/File:0025Pikachu.png "Pikachu")  Pikachu - [Cosplay Pikachu](/wiki/File:0025Pikachu-Cosplay.png "Cosplay Pikachu")  Cosplay Pikachu [Pikachu in a cap](/wiki/File:0025Pikachu-Original_Cap.png "Pikachu in a cap")  Pikachu in a cap [Partner Pikachu](/wiki/File:0025Pikachu-Partner.png "Partner Pikachu")  Partner Pikachu - [Partner Pikachu](/wiki/File:0025Pikachu-Partner.png "Partner Pikachu")  Partner Pikachu [Gigantamax Pikachu](/wiki/File:0025Pikachu-Gigantamax.png "Gigantamax Pikachu")  Gigantamax Pikachu [Pale Pikachu](/wiki/File:Pokopia_Peakychu.png "Pale Pikachu")  Pale Pikachu - [Images on the Bulbagarden Archives](https://archives.bulbagarden.net/wiki/Category:Pikachu "a:Category:Pikachu") | | | |
| --- | --- |
| **[Type](/wiki/Type "Type")**  - | [**Electric**](/wiki/Electric_(type) "Electric (type)") | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") | | --- | --- | | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") | | --- | --- | Cosplay Pikachu | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") | | --- | --- | Pikachu in a cap | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") | | --- | --- | Partner Pikachu | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") | | --- | --- | Gigantamax Pikachu | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") | | --- | --- | Pale Pikachu - | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") | | --- | --- | | |
| **[Abilities](/wiki/Ability "Ability")**  - [Static](/wiki/Static_(Ability) "Static (Ability)") [Cacophony](/wiki/Cacophony_(Ability) "Cacophony (Ability)")  Cosplay Pikachu [Cacophony](/wiki/Cacophony_(Ability) "Cacophony (Ability)")  Pikachu in a cap [Lightning Rod](/wiki/Lightning_Rod_(Ability) "Lightning Rod (Ability)")   Hidden Ability [Cacophony](/wiki/Cacophony_(Ability) "Cacophony (Ability)")   Hidden Ability [Cacophony](/wiki/Cacophony_(Ability) "Cacophony (Ability)")  Cosplay Pikachu [Cacophony](/wiki/Cacophony_(Ability) "Cacophony (Ability)")  Pikachu in a cap | |
| **[Gender ratio](/wiki/List_of_Pok%C3%A9mon_by_gender_ratio "List of Pokémon by gender ratio")**  - Unknown - [50% male, 50% female](/wiki/Category:Pok%C3%A9mon_with_a_gender_ratio_of_one_male_to_one_female "Category:Pokémon with a gender ratio of one male to one female") | **[Catch rate](/wiki/Catch_rate "Catch rate")**  | 190 (35.2%) | | --- | |
| **[Breeding](/wiki/Pok%C3%A9mon_breeding "Pokémon breeding")**  - **[Egg Groups](/wiki/Egg_Group "Egg Group")**  - [Field](/wiki/Field_(Egg_Group) "Field (Egg Group)") and [Fairy](/wiki/Fairy_(Egg_Group) "Fairy (Egg Group)") or [No Eggs Discovered](/wiki/No_Eggs_Discovered_(Egg_Group) "No Eggs Discovered (Egg Group)")[Cosplay](/wiki/Cosplay_Pikachu "Cosplay Pikachu")/[Cap](/wiki/Pikachu_in_a_cap "Pikachu in a cap") **[Hatch time](/wiki/Egg_cycle "Egg cycle")**  | 10 cycles | | --- | | |
| **[Height](/wiki/List_of_Pok%C3%A9mon_by_height "List of Pokémon by height")**  - 1'04" 0.4 m - Pikachu - 0'0" 0 m - Cosplay Pikachu - 0'0" 0 m - Pikachu in a cap - 0'0" 0 m - Partner Pikachu - 68'11"+ 21.0+ m - Gigantamax Pikachu - 1'04" 0.4 m - Pale Pikachu - 0'0" 0 m - {{{form7}}} | **[Weight](/wiki/Weight "Weight")**  - 13.2 lbs. 6.0 kg - Pikachu - 0 lbs. 0 kg - Cosplay Pikachu - 0 lbs. 0 kg - Pikachu in a cap - 0 lbs. 0 kg - Partner Pikachu - ????.? lbs. ???.? kg - Gigantamax Pikachu - 11.0 lbs. 5.0 kg - Pale Pikachu - 0 lbs. 0 kg - {{{form7}}} |
| **[Mega Stone](/wiki/Mega_Stone "Mega Stone")**  | [[\|]] | [[\|]] | | --- | --- | | |
| **[Base experience yield](/wiki/Experience "Experience")**  | 82   Gen. I-IV | Unknown   IV | 112   V+ | | --- | --- | --- | | **[Leveling rate](/wiki/Experience "Experience")**  | Medium Fast | | --- | |
| **[EV yield](/wiki/List_of_Pok%C3%A9mon_by_effort_value_yield "List of Pokémon by effort value yield")**  - Total: 2 - Pikachu - 0  HP 0  Atk 0  Def 0  Sp.Atk 0  Sp.Def 2  Speed - Cosplay Pikachu - 0  HP 0  Atk 0  Def 0  Sp.Atk 0  Sp.Def 0  Speed - Pikachu in a cap - 0  HP 0  Atk 0  Def 0  Sp.Atk 0  Sp.Def 0  Speed - Partner Pikachu - 0  HP 0  Atk 0  Def 0  Sp.Atk 0  Sp.Def 0  Speed | |
| **[Shape](/wiki/List_of_Pok%C3%A9mon_by_shape "List of Pokémon by shape")**  | [![](https://archives.bulbagarden.net/media/upload/thumb/c/cc/Body08.png/32px-Body08.png)](/wiki/File:Body08.png) | | --- | | **[Footprint](/wiki/Footprint "Footprint")**  | [![](https://archives.bulbagarden.net/media/upload/c/c4/F0025.png)](/wiki/File:F0025.png) | [![](https://archives.bulbagarden.net/media/upload/e/e3/None.png)](/wiki/File:None.png)   Cosplay Pikachu | | --- | --- | |
| **[Pokédex color](/wiki/List_of_Pok%C3%A9mon_by_color "List of Pokémon by color")**  | Yellow | | --- | | **[Base friendship](/wiki/List_of_Pok%C3%A9mon_by_base_friendship "List of Pokémon by base friendship")**  | 70 | | --- | |
| **External Links**  - On Smogon Pokédex: <br> [Generation I](https://www.smogon.com/dex/rb/pokemon/pikachu/) [Generation II](https://www.smogon.com/dex/gs/pokemon/pikachu/) [Generation III](https://www.smogon.com/dex/rs/pokemon/pikachu/) [Generation IV](https://www.smogon.com/dex/dp/pokemon/pikachu/) [Generation V](https://www.smogon.com/dex/bw/pokemon/pikachu/) [Generation VI](https://www.smogon.com/dex/xy/pokemon/pikachu/) [Generation VII](https://www.smogon.com/dex/sm/pokemon/pikachu/) [Generation VIII](https://www.smogon.com/dex/ss/pokemon/pikachu/) [Generation IX](https://www.smogon.com/dex/sv/pokemon/pikachu/) [Artwork on Bulbagarden Archives](https://archives.bulbagarden.net/wiki/Category:Pikachu "a:Category:Pikachu") | |


**Pikachu** ([Japanese](/wiki/List_of_Japanese_Pok%C3%A9mon_names "List of Japanese Pokémon names"): **ピカチュウ** *Pikachu*) is an [Electric-type](/wiki/Electric_(type) "Electric (type)") [Pokémon](/wiki/Pok%C3%A9mon_(species) "Pokémon (species)") introduced in [Generation I](/wiki/Generation_I "Generation I").

It [evolves](/wiki/Evolution "Evolution") from [Pichu](/wiki/Pichu_(Pok%C3%A9mon) "Pichu (Pokémon)") when [leveled up with high friendship](/wiki/Friendship_Evolution "Friendship Evolution") and evolves into [Raichu](/wiki/Raichu_(Pok%C3%A9mon) "Raichu (Pokémon)") when exposed to a [Thunder Stone](/wiki/Thunder_Stone "Thunder Stone").

In [Alola](/wiki/Alola "Alola"), Pikachu evolves into [Alolan](/wiki/Alolan_form "Alolan form") Raichu when exposed to a Thunder Stone.

Pikachu has sixteen alternate [forms](#Form_data) that fall into four groups: **[Cosplay Pikachu](/wiki/Cosplay_Pikachu "Cosplay Pikachu")**, **[Pikachu in a cap](/wiki/Pikachu_in_a_cap "Pikachu in a cap")**, the **[partner](/wiki/Partner_Pok%C3%A9mon_(Let%27s_Go,_Pikachu!_and_Let%27s_Go,_Eevee!) "Partner Pokémon (Let's Go, Pikachu! and Let's Go, Eevee!)") Pikachu**, and **Gigantamax Pikachu**. Ordinary Pikachu can [Gigantamax](/wiki/Gigantamax "Gigantamax") into Gigantamax Pikachu if it has the [Gigantamax Factor](/wiki/Gigantamax#Gigantamax_Factor "Gigantamax"). Additionally, many other [Pikachu variants](/wiki/Pikachu_variants "Pikachu variants") have appeared in various media.

Cosplay Pikachu, Pikachu in a cap, the partner Pikachu, and Pikachu with the Gigantamax Factor cannot evolve. The [Pikachu](/wiki/Pikachu_(Yellow) "Pikachu (Yellow)") received at the beginning of [Pokémon Yellow Version](/wiki/Pok%C3%A9mon_Yellow_Version "Pokémon Yellow Version") will refuse to evolve into Raichu unless it is [traded](/wiki/Trade "Trade") and evolved on another save file.

Pikachu is popularly known as the mascot of the [Pokémon](/wiki/Pok%C3%A9mon "Pokémon") franchise and one of [Nintendo](/wiki/Nintendo "Nintendo")'s major mascots. It is also the [game mascot](/wiki/Game_mascot "Game mascot") and the [player's first Pokémon](/wiki/List_of_the_player%27s_first_Pok%C3%A9mon "List of the player's first Pokémon") in [Pokémon: Yellow Version: Special Pikachu Edition](/wiki/Pok%C3%A9mon_Yellow_Version "Pokémon Yellow Version") and [Let's Go, Pikachu!](/wiki/Pok%C3%A9mon:_Let%27s_Go,_Pikachu!_and_Let%27s_Go,_Eevee! "Pokémon: Let's Go, Pikachu! and Let's Go, Eevee!"), as well as in [Pokémon Rumble Blast](/wiki/Pok%C3%A9mon_Rumble_Blast "Pokémon Rumble Blast") and [Pokémon Rumble World](/wiki/Pok%C3%A9mon_Rumble_World "Pokémon Rumble World"), and has made numerous appearances on the covers of spin-off games.

## Contents

- [1 Biology](#Biology)
  - [1.1 Forms](#Forms)
    - [1.1.1 Cosplay Pikachu](#Cosplay_Pikachu)
    - [1.1.2 Pikachu in a cap](#Pikachu_in_a_cap)
    - [1.1.3 Partner Pikachu](#Partner_Pikachu)
    - [1.1.4 Gigantamax Pikachu](#Gigantamax_Pikachu)
    - [1.1.5 Mr. Windychu & Ms. Wavychu](#Mr._Windychu_&_Ms._Wavychu)
  - [1.2 Evolution](#Evolution)
- [2 Game data](#Game_data)
  - [2.1 As the player](#As_the_player)
  - [2.2 NPC appearances](#NPC_appearances)
    - [2.2.1 In the core series](#In_the_core_series)
    - [2.2.2 In the spin-off games](#In_the_spin-off_games)
  - [2.3 Pokédex entries](#Pok%C3%A9dex_entries)
  - [2.4 Game locations](#Game_locations)
    - [2.4.1 In side games](#In_side_games)
    - [2.4.2 In events](#In_events)
      - [2.4.2.1 Pikachu](#Pikachu)
      - [2.4.2.2 Original Cap Pikachu](#Original_Cap_Pikachu)
      - [2.4.2.3 Hoenn Cap Pikachu](#Hoenn_Cap_Pikachu)
      - [2.4.2.4 Sinnoh Cap Pikachu](#Sinnoh_Cap_Pikachu)
      - [2.4.2.5 Unova Cap Pikachu](#Unova_Cap_Pikachu)
      - [2.4.2.6 Kalos Cap Pikachu](#Kalos_Cap_Pikachu)
      - [2.4.2.7 Alola Cap Pikachu](#Alola_Cap_Pikachu)
      - [2.4.2.8 Partner Cap Pikachu](#Partner_Cap_Pikachu)
      - [2.4.2.9 World Cap Pikachu](#World_Cap_Pikachu)
    - [2.4.3 In-game events](#In-game_events)
      - [2.4.3.1 Pikachu](#Pikachu_2)
      - [2.4.3.2 Partner Cap Pikachu](#Partner_Cap_Pikachu_2)
    - [2.4.4 Wild Area News](#Wild_Area_News)
    - [2.4.5 Poké Portal News](#Pok%C3%A9_Portal_News)
    - [2.4.6 Pokémon Global Link promotions](#Pok%C3%A9mon_Global_Link_promotions)
  - [2.5 Held items](#Held_items)
  - [2.6 Stats](#Stats)
    - [2.6.1 Base stats](#Base_stats)
      - [2.6.1.1 Generations I-V](#Generations_I-V)
      - [2.6.1.2 Generation VI onward](#Generation_VI_onward)
      - [2.6.1.3 Partner Pikachu](#Partner_Pikachu_2)
    - [2.6.2 Pokéathlon stats](#Pok%C3%A9athlon_stats)
  - [2.7 Type effectiveness](#Type_effectiveness)
  - [2.8 Learnset](#Learnset)
    - [2.8.1 By leveling up](#By_leveling_up)
    - [2.8.2 By TM](#By_TM)
    - [2.8.3 By breeding](#By_breeding)
    - [2.8.4 By a prior Evolution](#By_a_prior_Evolution)
    - [2.8.5 By events](#By_events)
    - [2.8.6 TCG-only moves](#TCG-only_moves)
    - [2.8.7 Animated series-only moves](#Animated_series-only_moves)
  - [2.9 Side game data](#Side_game_data)
    - [2.9.1 Pikachu](#Pikachu_3)
    - [2.9.2 Pikachu Libre](#Pikachu_Libre)
    - [2.9.3 Pikachu Pop Star](#Pikachu_Pop_Star)
    - [2.9.4 Pikachu Rock Star](#Pikachu_Rock_Star)
    - [2.9.5 Flying Pikachu](#Flying_Pikachu)
    - [2.9.6 Shaymin scarf Pikachu](#Shaymin_scarf_Pikachu)
    - [2.9.7 Pikachu, Ph. D](#Pikachu,_Ph._D)
    - [2.9.8 Captain Pikachu](#Captain_Pikachu)
    - [2.9.9 Gigantamax Pikachu](#Gigantamax_Pikachu_2)
  - [2.10 Form data](#Form_data)
    - [2.10.1 Cosplay Pikachu](#Cosplay_Pikachu_2)
    - [2.10.2 Pikachu in a cap](#Pikachu_in_a_cap_2)
    - [2.10.3 Partner Pikachu](#Partner_Pikachu_3)
    - [2.10.4 Gigantamax](#Gigantamax)
  - [2.11 Evolution data](#Evolution_data)
  - [2.12 Sprites](#Sprites)
    - [2.12.1 Other sprites](#Other_sprites)
- [3 In animation](#In_animation)
  - [3.1 Main series](#Main_series)
    - [3.1.1 Major appearances](#Major_appearances)
      - [3.1.1.1 Ash's Pikachu](#Ash's_Pikachu)
      - [3.1.1.2 Pikachutwo](#Pikachutwo)
      - [3.1.1.3 Puka](#Puka)
      - [3.1.1.4 Sparky](#Sparky)
      - [3.1.1.5 Ashachu](#Ashachu)
      - [3.1.1.6 Cosplay Pikachu](#Cosplay_Pikachu_3)
      - [3.1.1.7 Ash's Pikachu (M20)](#Ash's_Pikachu_(M20))
      - [3.1.1.8 Goh's Pikachu](#Goh's_Pikachu)
      - [3.1.1.9 Captain Pikachu](#Captain_Pikachu_2)
      - [3.1.1.10 Other](#Other)
    - [3.1.2 Minor appearances](#Minor_appearances)
    - [3.1.3 Pokédex entries](#Pok%C3%A9dex_entries_2)
  - [3.2 Pokémon Mystery Dungeon Animated Shorts](#Pok%C3%A9mon_Mystery_Dungeon_Animated_Shorts)
  - [3.3 Pokémon Origins](#Pok%C3%A9mon_Origins)
    - [3.3.1 Red's Pikachu](#Red's_Pikachu)
    - [3.3.2 Other](#Other_2)
  - [3.4 Pokémon Generations](#Pok%C3%A9mon_Generations)
    - [3.4.1 Red's Pikachu](#Red's_Pikachu_2)
  - [3.5 Pokémon Masters Animated Trailer](#Pok%C3%A9mon_Masters_Animated_Trailer)
  - [3.6 Pokémon: Twilight Wings](#Pok%C3%A9mon:_Twilight_Wings)
  - [3.7 POKÉTOON](#POK%C3%89TOON)
  - [3.8 GOTCHA!](#GOTCHA!)
  - [3.9 Pokémon Evolutions](#Pok%C3%A9mon_Evolutions)
  - [3.10 Bidoof's Big Stand](#Bidoof's_Big_Stand)
  - [3.11 A Ripple in Time](#A_Ripple_in_Time)
  - [3.12 Pokémon Concierge](#Pok%C3%A9mon_Concierge)
  - [3.13 The Journey of One Dream](#The_Journey_of_One_Dream)
  - [3.14 Biri-Biri](#Biri-Biri)
- [4 In the manga](#In_the_manga)
  - [4.1 Pokémon Adventures](#Pok%C3%A9mon_Adventures)
    - [4.1.1 Major appearances](#Major_appearances_2)
      - [4.1.1.1 Pika](#Pika)
      - [4.1.1.2 Chuchu](#Chuchu)
      - [4.1.1.3 Cosplay Pikachu](#Cosplay_Pikachu_4)
      - [4.1.1.4 Other](#Other_3)
    - [4.1.2 Minor appearances](#Minor_appearances_2)
    - [4.1.3 Pokédex entries](#Pok%C3%A9dex_entries_3)
  - [4.2 Pokémon Pocket Monsters](#Pok%C3%A9mon_Pocket_Monsters)
  - [4.3 The Electric Tale of Pikachu](#The_Electric_Tale_of_Pikachu)
    - [4.3.1 Pokédex entries](#Pok%C3%A9dex_entries_4)
  - [4.4 Magical Pokémon Journey and Pokémon Chamo-Chamo ☆ Pretty ♪](#Magical_Pok%C3%A9mon_Journey_and_Pok%C3%A9mon_Chamo-Chamo_%E2%98%86_Pretty_%E2%99%AA)
  - [4.5 Pokémon Zensho](#Pok%C3%A9mon_Zensho)
  - [4.6 Pokémon: Yeah! I Got Pokémon!](#Pok%C3%A9mon:_Yeah!_I_Got_Pok%C3%A9mon!)
  - [4.7 How I Became a Pokémon Card](#How_I_Became_a_Pok%C3%A9mon_Card)
  - [4.8 Pokémon Gold & Silver: The Golden Boys](#Pok%C3%A9mon_Gold_&_Silver:_The_Golden_Boys)
  - [4.9 Pokémon Newspaper Strip](#Pok%C3%A9mon_Newspaper_Strip)
  - [4.10 Ash & Pikachu](#Ash_&_Pikachu)
  - [4.11 Pokémon Battle Frontier](#Pok%C3%A9mon_Battle_Frontier)
  - [4.12 Pocket Monsters Diamond & Pearl](#Pocket_Monsters_Diamond_&_Pearl)
  - [4.13 Pokémon Battrio: Aim to be Battrio Master!](#Pok%C3%A9mon_Battrio:_Aim_to_be_Battrio_Master!)
  - [4.14 Pocket Monsters HeartGold & SoulSilver Go! Go! Pokéathlon](#Pocket_Monsters_HeartGold_&_SoulSilver_Go!_Go!_Pok%C3%A9athlon)
  - [4.15 Pocket Monsters HGSS Jō's Big Adventure](#Pocket_Monsters_HGSS_J%C5%8D's_Big_Adventure)
  - [4.16 Pokémon + Nobunaga's Ambition ~ Ranse's Color Picture Scroll ~](#Pok%C3%A9mon_+_Nobunaga's_Ambition_~_Ranse's_Color_Picture_Scroll_~)
  - [4.17 Pokémon Horizon](#Pok%C3%A9mon_Horizon)
  - [4.18 Pokémon Journeys](#Pok%C3%A9mon_Journeys)
- [5 In the TCG](#In_the_TCG)
- [6 In the TFG](#In_the_TFG)
- [7 Other appearances](#Other_appearances)
  - [7.1 Pokkén Tournament](#Pokk%C3%A9n_Tournament)
  - [7.2 Super Smash Bros.](#Super_Smash_Bros.)
  - [7.3 Detective Pikachu](#Detective_Pikachu)
  - [7.4 Pokémon UNITE](#Pok%C3%A9mon_UNITE)
  - [7.5 The Strength of the Lightning](#The_Strength_of_the_Lightning)
  - [7.6 Celestial](#Celestial)
- [8 Trivia](#Trivia)
  - [8.1 Merchandise](#Merchandise)
  - [8.2 Real life](#Real_life)
  - [8.3 Cultural impact](#Cultural_impact)
  - [8.4 Concept and development](#Concept_and_development)
  - [8.5 Origin](#Origin)
    - [8.5.1 Name origin](#Name_origin)
- [9 In other languages](#In_other_languages)
- [10 See also](#See_also)
- [11 References](#References)
- [12 External links](#External_links)

## Biology

[![](https://archives.bulbagarden.net/media/upload/thumb/0/05/HOME0845Go.png/200px-HOME0845Go.png)](/wiki/File:HOME0845Go.png)A Pikachu who has been accidentally partially [swallowed](/wiki/Gulp_Missile_(Ability) "Gulp Missile (Ability)") by a [Cramorant](/wiki/Cramorant_(Pok%C3%A9mon) "Cramorant (Pokémon)")

Pikachu is a short, chubby [rodent](https://en.wikipedia.org/wiki/rodent "wp:rodent") [Pokémon](/wiki/Pok%C3%A9mon_(species) "Pokémon (species)"). It is covered in yellow fur with two horizontal brown stripes on its back. It has a small mouth, long, pointed ears with black tips, and brown eyes. Each cheek is a red circle that contains a pouch for electricity storage. It has short forearms with five fingers on each paw, and its feet each have three toes. At the base of its lightning bolt-shaped tail is a patch of brown fur. A [female](/wiki/List_of_Pok%C3%A9mon_with_gender_differences "List of Pokémon with gender differences") Pikachu has a "V"-shaped notch at the end of its tail, which looks like the top of a heart. It is classified as a quadruped, but it has been known to stand and walk on its hind legs; therefore, Pikachu is a facultative biped.

Pikachu stores electric energy using the small pouches in its cheeks in case it is attacked, so one might feel a small shock if it is touched. These pouches become electrically charged during the night as it sleeps, so a dozy Pikachu may occasionally discharge electricity after waking up. If it does not get enough sleep, it will not be able to release said electricity at its full power. When angered or threatened, Pikachu quickly unleashes the electric energy from its cheeks, which can reach the intensity of a lightning bolt. This also serves as a sign that it's wary, and it will feel stressed if it's unable to fully release the electricity. In *[Sparks Fly for Magnemite](/wiki/EP030 "EP030")*, Pikachu is shown to accumulate energy in its glands, which it will need to eject to avoid complications. When several Pikachu form a group, they are capable of causing thunder storms by building up electricity. If one Pikachu sees another one who is weakened and in need of help, it revitalizes it by recharging it with an electric shock. However, as seen with [Peakychu](/wiki/Peakychu "Peakychu"), overusing this ability alters its physical structure, causing it to lose its ability to generate electricity and its fur to become pale. In [Alola](/wiki/Alola "Alola"), a plan was recently announced to help build an electric power plant by gathering many Pikachu. If its cheek sacs are very soft and stretchy, this means that it can generate powerful electricity. As shown in [Pokémon Sleep](/wiki/Pok%C3%A9mon_Sleep "Pokémon Sleep"), Pikachu is known to emit electricity through its cheeks while sleeping, perhaps because it's dreaming of firing electric shots.[[1]](#cite_note-1)

Pikachu is a curious, playful, and intelligent Pokémon who primarily eats fruits and roasts [Berries](/wiki/Berry "Berry") with its electricity to make them tender enough to eat. However, 

*... (truncated) ...*

## 5. Fetch Wikipedia — Clean content extraction

In [7]:
args = FetchPageInput(
    url="https://en.wikipedia.org/wiki/Pok%C3%A9mon",
    css_selector="#bodyContent",
)
raw = fetch_page_as_markdown(args)
result = PageMarkdownResult.model_validate_json(raw)

print(f"Title: {result.title}")
print(f"Error: {result.error}")
print(f"Markdown length: {len(result.markdown):,} chars")
print("---")
display(Markdown(result.markdown[:3000] + "\n\n*... (truncated) ...*"))

[2026-03-19 17:08:25] INFO: Fetched (200) <GET https://en.wikipedia.org/wiki/Pok%C3%A9mon> (referer: https://www.google.com/)


Title: Pokémon - Wikipedia
Error: None
Markdown length: 302,460 chars
---


[![Page semi-protected](//upload.wikimedia.org/wikipedia/en/thumb/1/1b/Semi-protection-shackle.svg/20px-Semi-protection-shackle.svg.png)](/wiki/Wikipedia:Protection_policy#semi "This article is semi-protected.")

From Wikipedia, the free encyclopedia

Japanese media franchise

This article is about the media franchise as a whole. For the video game series, see [*Pokémon* (video game series)](/wiki/Pok%C3%A9mon_(video_game_series) "Pokémon (video game series)"). For the animated series, see [*Pokémon* (TV series)](/wiki/Pok%C3%A9mon_(TV_series) "Pokémon (TV series)"). For a list of creatures known as "Pokémon", see [List of Pokémon](/wiki/List_of_Pok%C3%A9mon "List of Pokémon"). For other uses, see [Pokémon (disambiguation)](/wiki/Pok%C3%A9mon_(disambiguation) "Pokémon (disambiguation)").

| Pokémon | |
| --- | --- |
| [![](//upload.wikimedia.org/wikipedia/commons/thumb/9/98/International_Pok%C3%A9mon_logo.svg/330px-International_Pok%C3%A9mon_logo.svg.png)](/wiki/File:International_Pok%C3%A9mon_logo.svg)   International franchise logo | |
| Created by | [Satoshi Tajiri](/wiki/Satoshi_Tajiri "Satoshi Tajiri") |
| Original work | [*Pocket Monsters Red* and *Pocket Monsters Green*](/wiki/Pok%C3%A9mon_Red,_Blue,_and_Yellow "Pokémon Red, Blue, and Yellow") (1996) |
| Owners | <br>[Nintendo](/wiki/Nintendo "Nintendo")[Creatures](/wiki/Creatures_(company) "Creatures (company)")[Game Freak](/wiki/Game_Freak "Game Freak")[[1]](#cite_note-pokemon.com-1) |
| Years | 1996–present |
| Print publications | |
| Comics | See [list of *Pokémon* manga](/wiki/List_of_Pok%C3%A9mon_manga "List of Pokémon manga") |
| Films and television | |
| Film(s) | See [list of *Pokémon* films](/wiki/List_of_Pok%C3%A9mon_films "List of Pokémon films") |
| Animated series | [*Pokémon*](/wiki/Pok%C3%A9mon_(TV_series) "Pokémon (TV series)") (1997–present) |
| Games | |
| Traditional | *[Pokémon Trading Card Game](/wiki/Pok%C3%A9mon_Trading_Card_Game "Pokémon Trading Card Game")* |
| Video game(s) | [*Pokémon* video game series](/wiki/Pok%C3%A9mon_(video_game_series) "Pokémon (video game series)") |
| Official website | |
| [Official hub](https://www.portal-pokemon.com/) | |


***Pokémon***[[a]](#cite_note-2)[[b]](#cite_note-3) is a Japanese [media franchise](/wiki/Media_franchise "Media franchise") consisting of [video games](/wiki/List_of_Pok%C3%A9mon_video_games "List of Pokémon video games"), [animated series](/wiki/Pok%C3%A9mon_(TV_series) "Pokémon (TV series)") and [films](/wiki/List_of_Pok%C3%A9mon_films "List of Pokémon films"), [a trading card game](/wiki/Pok%C3%A9mon_Trading_Card_Game "Pokémon Trading Card Game"), and other related media. The franchise takes place in a [shared universe](/wiki/Shared_universe "Shared universe") in which humans co-exist with creatures known as [Pokémon](/wiki/List_of_Pok%C3%A9mon "List of Pokémon"), a large variety of species endowed with special powers. The franchise's primary [target audience](/wiki/Target_audience "Target audience") is chil

*... (truncated) ...*

## 6. Search → Fetch pipeline

Search for something, then fetch the first result and show its markdown.

In [9]:
# Step 1: Search
search_args = GoogleSearchInput(
    query="Bulbasaur",
    site_restrict="bulbapedia.bulbagarden.net",
    max_results=1,
)
search_raw = google_search(search_args)
search_result = GoogleSearchResult.model_validate_json(search_raw)

if search_result.results:
    first = search_result.results[0]
    print(f"Top result: {first.title}")
    print(f"URL: {first.url}\n")

    # Step 2: Fetch that page
    fetch_args = FetchPageInput(
        url=first.url,
        css_selector="#mw-content-text",
        use_stealth=True,
    )
    fetch_raw = fetch_page_as_markdown(fetch_args)
    page = PageMarkdownResult.model_validate_json(fetch_raw)

    print(f"Page title: {page.title}")
    print(f"Content length: {len(page.markdown):,} chars")
    print("---")
    display(Markdown(page.markdown[:20000] + "\n\n*... (truncated) ...*"))
else:
    print("No results found.")

[2026-03-19 17:10:01] INFO: Fetched (200) <GET https://www.google.com/search?q=site%3Abulbapedia.bulbagarden.net+Bulbasaur&num=1&hl=en&sei=2B-8abnnHYCJxc8PoYOY6Q4> (referer: https://www.google.com/)


Top result: Bulbasaur (Pokémon) - Bulbapedia
URL: https://bulbapedia.bulbagarden.net/wiki/Bulbasaur_(Pok%C3%A9mon)



[2026-03-19 17:10:04] INFO: Fetched (200) <GET https://bulbapedia.bulbagarden.net/wiki/Bulbasaur_(Pok%C3%A9mon)> (referer: https://www.google.com/)


Page title: Bulbasaur (Pokémon) - Bulbapedia, the community-driven Pokémon encyclopedia
Content length: 194,247 chars
---




- For Pokémon GO information on this species, see [the game's section](#Pok%C3%A9mon_GO).
- | [Pokémon](/wiki/List_of_Pok%C3%A9mon_by_National_Pok%C3%A9dex_number "List of Pokémon by National Pokédex number") |
| --- | - [#0002: Ivysaur](/wiki/Ivysaur_(Pok%C3%A9mon) "Ivysaur (Pokémon)") [/wiki/Ivysaur_(Pok%C3%A9mon)](/wiki/Ivysaur_(Pok%C3%A9mon) "Ivysaur (Pokémon)") [→](/wiki/Ivysaur_(Pok%C3%A9mon) "Ivysaur (Pokémon)")
- This article is about the species. For a specific instance of this species, see [Bulbasaur (disambiguation)](/wiki/Bulbasaur_(disambiguation) "Bulbasaur (disambiguation)").

| | | **Bulbasaur**   [Seed Pokémon](/wiki/Pok%C3%A9mon_category "Pokémon category") | **フシギダネ**   *Fushigidane* | | --- | --- | | [#0001](/wiki/List_of_Pok%C3%A9mon_by_National_Pok%C3%A9dex_number "List of Pokémon by National Pokédex number") | | --- | --- | | - [Bulbasaur](/wiki/File:0001Bulbasaur.png "Bulbasaur") - [Images on the Bulbagarden Archives](https://archives.bulbagarden.net/wiki/Category:Bulbasaur "a:Category:Bulbasaur") | | | |
| --- | --- |
| **[Type](/wiki/Type "Type")**  - | [**Grass**](/wiki/Grass_(type) "Grass (type)") | [**Poison**](/wiki/Poison_(type) "Poison (type)") | | --- | --- | | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") | | --- | --- | | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") | | --- | --- | | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") | | --- | --- | | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") | | --- | --- | | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") | | --- | --- | - | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") | | --- | --- | | |
| **[Abilities](/wiki/Ability "Ability")**  - [Overgrow](/wiki/Overgrow_(Ability) "Overgrow (Ability)") [Cacophony](/wiki/Cacophony_(Ability) "Cacophony (Ability)") [Cacophony](/wiki/Cacophony_(Ability) "Cacophony (Ability)") [Chlorophyll](/wiki/Chlorophyll_(Ability) "Chlorophyll (Ability)")   Hidden Ability [Cacophony](/wiki/Cacophony_(Ability) "Cacophony (Ability)")   Hidden Ability [Cacophony](/wiki/Cacophony_(Ability) "Cacophony (Ability)") [Cacophony](/wiki/Cacophony_(Ability) "Cacophony (Ability)") | |
| **[Gender ratio](/wiki/List_of_Pok%C3%A9mon_by_gender_ratio "List of Pokémon by gender ratio")**  - Unknown - [87.5% male, 12.5% female](/wiki/Category:Pok%C3%A9mon_with_a_gender_ratio_of_seven_males_to_one_female "Category:Pokémon with a gender ratio of seven males to one female") | **[Catch rate](/wiki/Catch_rate "Catch rate")**  | 45 (11.9%) | | --- | |
| **[Breeding](/wiki/Pok%C3%A9mon_breeding "Pokémon breeding")**  - **[Egg Groups](/wiki/Egg_Group "Egg Group")**  | [Monster](/wiki/Monster_(Egg_Group) "Monster (Egg Group)") and [Grass](/wiki/Grass_(Egg_Group) "Grass (Egg Group)") | | --- | **[Hatch time](/wiki/Egg_cycle "Egg cycle")**  | 20 cycles | | --- | | |
| **[Height](/wiki/List_of_Pok%C3%A9mon_by_height "List of Pokémon by height")**  - 2'04" 0.7 m - Bulbasaur - 0'0" 0 m - {{{form2}}} - 0'0" 0 m - {{{form3}}} - 0'0" 0 m - {{{form4}}} - 0'0" 0 m - {{{form5}}} - 0'0" 0 m - {{{form6}}} - 0'0" 0 m - {{{form7}}} | **[Weight](/wiki/Weight "Weight")**  - 15.2 lbs. 6.9 kg - Bulbasaur - 0 lbs. 0 kg - {{{form2}}} - 0 lbs. 0 kg - {{{form3}}} - 0 lbs. 0 kg - {{{form4}}} - 0 lbs. 0 kg - {{{form5}}} - 0 lbs. 0 kg - {{{form6}}} - 0 lbs. 0 kg - {{{form7}}} |
| **[Mega Stone](/wiki/Mega_Stone "Mega Stone")**  | [[\|]] | [[\|]] | | --- | --- | | |
| **[Base experience yield](/wiki/Experience "Experience")**  | 64   Gen. I-IV | Unknown   IV | 64   V+ | | --- | --- | --- | | **[Leveling rate](/wiki/Experience "Experience")**  | Medium Slow | | --- | |
| **[EV yield](/wiki/List_of_Pok%C3%A9mon_by_effort_value_yield "List of Pokémon by effort value yield")**  - Total: 1 - Bulbasaur - 0  HP 0  Atk 0  Def 1  Sp.Atk 0  Sp.Def 0  Speed - 0  HP 0  Atk 0  Def 0  Sp.Atk 0  Sp.Def 0  Speed - 0  HP 0  Atk 0  Def 0  Sp.Atk 0  Sp.Def 0  Speed - 0  HP 0  Atk 0  Def 0  Sp.Atk 0  Sp.Def 0  Speed | |
| **[Shape](/wiki/List_of_Pok%C3%A9mon_by_shape "List of Pokémon by shape")**  | [![](https://archives.bulbagarden.net/media/upload/thumb/c/cc/Body08.png/32px-Body08.png)](/wiki/File:Body08.png) | | --- | | **[Footprint](/wiki/Footprint "Footprint")**  | [![](https://archives.bulbagarden.net/media/upload/d/d1/F0001.png)](/wiki/File:F0001.png) | [![](https://archives.bulbagarden.net/media/upload/e/e3/None.png)](/wiki/File:None.png)   {{{form2}}} | | --- | --- | |
| **[Pokédex color](/wiki/List_of_Pok%C3%A9mon_by_color "List of Pokémon by color")**  | Green | | --- | | **[Base friendship](/wiki/List_of_Pok%C3%A9mon_by_base_friendship "List of Pokémon by base friendship")**  | 70 | | --- | |
| **External Links**  - On Smogon Pokédex: <br> [Generation I](https://www.smogon.com/dex/rb/pokemon/bulbasaur/) [Generation II](https://www.smogon.com/dex/gs/pokemon/bulbasaur/) [Generation III](https://www.smogon.com/dex/rs/pokemon/bulbasaur/) [Generation IV](https://www.smogon.com/dex/dp/pokemon/bulbasaur/) [Generation V](https://www.smogon.com/dex/bw/pokemon/bulbasaur/) [Generation VI](https://www.smogon.com/dex/xy/pokemon/bulbasaur/) [Generation VII](https://www.smogon.com/dex/sm/pokemon/bulbasaur/) [Generation VIII](https://www.smogon.com/dex/ss/pokemon/bulbasaur/) [Generation IX](https://www.smogon.com/dex/sv/pokemon/bulbasaur/) [Artwork on Bulbagarden Archives](https://archives.bulbagarden.net/wiki/Category:Bulbasaur "a:Category:Bulbasaur") | |


**Bulbasaur** ([Japanese](/wiki/List_of_Japanese_Pok%C3%A9mon_names "List of Japanese Pokémon names"): **フシギダネ** *Fushigidane*) is a dual-type [Grass](/wiki/Grass_(type) "Grass (type)")/[Poison](/wiki/Poison_(type) "Poison (type)") [Pokémon](/wiki/Pok%C3%A9mon_(species) "Pokémon (species)") introduced in [Generation I](/wiki/Generation_I "Generation I").

It [evolves](/wiki/Evolution "Evolution") into [Ivysaur](/wiki/Ivysaur_(Pok%C3%A9mon) "Ivysaur (Pokémon)") starting at [level](/wiki/Level "Level") 16, which evolves into [Venusaur](/wiki/Venusaur_(Pok%C3%A9mon) "Venusaur (Pokémon)") starting at level 32.

Along with [Charmander](/wiki/Charmander_(Pok%C3%A9mon) "Charmander (Pokémon)") and [Squirtle](/wiki/Squirtle_(Pok%C3%A9mon) "Squirtle (Pokémon)"), Bulbasaur is one of the three [first partner Pokémon of Kanto](/wiki/Kanto_first_partner_Pok%C3%A9mon "Kanto first partner Pokémon") available at the beginning of [Pokémon Red, Green](/wiki/Pok%C3%A9mon_Red_and_Green_Versions "Pokémon Red and Green Versions"), [Blue](/wiki/Pok%C3%A9mon_Red_and_Blue_Versions "Pokémon Red and Blue Versions"), [FireRed, and LeafGreen](/wiki/Pok%C3%A9mon_FireRed_and_LeafGreen_Versions "Pokémon FireRed and LeafGreen Versions").

## Contents

- [1 Biology](#Biology)
  - [1.1 Evolution](#Evolution)
- [2 Game data](#Game_data)
  - [2.1 Pokédex entries](#Pok%C3%A9dex_entries)
  - [2.2 Game locations](#Game_locations)
    - [2.2.1 In side games](#In_side_games)
    - [2.2.2 In events](#In_events)
      - [2.2.2.1 In-game events](#In-game_events)
      - [2.2.2.2 Wild Area News](#Wild_Area_News)
    - [2.2.3 Pokémon Global Link promotions](#Pok%C3%A9mon_Global_Link_promotions)
  - [2.3 Held items](#Held_items)
  - [2.4 Stats](#Stats)
    - [2.4.1 Base stats](#Base_stats)
    - [2.4.2 Pokéathlon stats](#Pok%C3%A9athlon_stats)
  - [2.5 Type effectiveness](#Type_effectiveness)
  - [2.6 Learnset](#Learnset)
    - [2.6.1 By leveling up](#By_leveling_up)
    - [2.6.2 By TM](#By_TM)
    - [2.6.3 By breeding](#By_breeding)
    - [2.6.4 TCG-only moves](#TCG-only_moves)
    - [2.6.5 Animated series-only moves](#Animated_series-only_moves)
  - [2.7 Side game data](#Side_game_data)
  - [2.8 Evolution data](#Evolution_data)
  - [2.9 Sprites](#Sprites)
- [3 In animation](#In_animation)
  - [3.1 Main series](#Main_series)
    - [3.1.1 Major appearances](#Major_appearances)
      - [3.1.1.1 Ash's Bulbasaur](#Ash's_Bulbasaur)
      - [3.1.1.2 May's Bulbasaur](#May's_Bulbasaur)
      - [3.1.1.3 Shauna's Bulbasaur](#Shauna's_Bulbasaur)
      - [3.1.1.4 Other](#Other)
    - [3.1.2 Minor appearances](#Minor_appearances)
    - [3.1.3 Pokédex entries](#Pok%C3%A9dex_entries_2)
  - [3.2 Pokémon Origins](#Pok%C3%A9mon_Origins)
  - [3.3 Pokémon Generations](#Pok%C3%A9mon_Generations)
  - [3.4 Pokémon: Twilight Wings](#Pok%C3%A9mon:_Twilight_Wings)
  - [3.5 GOTCHA!](#GOTCHA!)
  - [3.6 POKÉTOON](#POK%C3%89TOON)
  - [3.7 Pokémon Evolutions](#Pok%C3%A9mon_Evolutions)
  - [3.8 Pokémon Concierge](#Pok%C3%A9mon_Concierge)
- [4 In the manga](#In_the_manga)
  - [4.1 Pokémon Adventures](#Pok%C3%A9mon_Adventures)
    - [4.1.1 Major appearances](#Major_appearances_2)
      - [4.1.1.1 Saur](#Saur)
    - [4.1.2 Minor appearances](#Minor_appearances_2)
  - [4.2 Pokémon Pocket Monsters](#Pok%C3%A9mon_Pocket_Monsters)
  - [4.3 The Electric Tale of Pikachu](#The_Electric_Tale_of_Pikachu)
  - [4.4 Magical Pokémon Journey](#Magical_Pok%C3%A9mon_Journey)
  - [4.5 Pokémon Zensho](#Pok%C3%A9mon_Zensho)
  - [4.6 Movie adaptations](#Movie_adaptations)
  - [4.7 Pokémon: Yeah! I Got Pokémon!](#Pok%C3%A9mon:_Yeah!_I_Got_Pok%C3%A9mon!)
  - [4.8 Pokémon Gold & Silver: The Golden Boys](#Pok%C3%A9mon_Gold_&_Silver:_The_Golden_Boys)
  - [4.9 Ash & Pikachu](#Ash_&_Pikachu)
- [5 In the TCG](#In_the_TCG)
- [6 In the TFG](#In_the_TFG)
- [7 Other appearances](#Other_appearances)
  - [7.1 Super Smash Bros. Melee and Brawl](#Super_Smash_Bros._Melee_and_Brawl)
    - [7.1.1 Melee trophy information](#Melee_trophy_information)
    - [7.1.2 Brawl trophy information](#Brawl_trophy_information)
  - [7.2 *POKÉMON Detective Pikachu*](#POK%C3%89MON_Detective_Pikachu)
  - [7.3 Celestial](#Celestial)
- [8 Trivia](#Trivia)
  - [8.1 Cultural impact](#Cultural_impact)
  - [8.2 Concept and development](#Concept_and_development)
  - [8.3 Origin](#Origin)
    - [8.3.1 Name origin](#Name_origin)
- [9 In other languages](#In_other_languages)
- [10 See also](#See_also)
- [11 References](#References)
- [12 External links](#External_links)

## Biology

[![](https://archives.bulbagarden.net/media/upload/thumb/e/e9/Art_Life_20230116_Bulbasaur.jpg/220px-Art_Life_20230116_Bulbasaur.jpg)](/wiki/File:Art_Life_20230116_Bulbasaur.jpg)Bulbasaur using its vines to climb a tree

Bulbasaur is a small, quadrupedal [amphibian](https://en.wikipedia.org/wiki/amphibian "wp:amphibian") [Pokémon](/wiki/Pok%C3%A9mon_(species) "Pokémon (species)") that has blue-green skin with darker patches. It has red eyes with white pupils, pointed, ear-like structures on top of its head, and a short, blunt snout with a wide mouth. Small, pointed teeth are visible in the upper jaw when the mouth is open. Each of its thick legs ends with three sharp claws. On Bulbasaur's back is a green plant bulb that conceals two slender, tentacle-like vines, which grow from a seed planted there at birth. The bulb also provides it with energy through photosynthesis and from the nutrient-rich seeds contained within.

Bulbasaur is sometimes seen seen napping in bright sunlight and can survive for days without eating. It is found in [grasslands](/wiki/List_of_Pok%C3%A9mon_by_habitat#Grassland_Pok%C3%A9mon "List of Pokémon by habitat") and forests throughout the [Kanto](/wiki/Kanto "Kanto") [region](/wiki/Region "Region"). However, due to its status as a [first partner Pokémon](/wiki/First_partner_Pok%C3%A9mon "First partner Pokémon"), it is hard to come by in the wild and is generally found under the ownership of a Trainer. It has been recently seen living in the [Terarium](/wiki/Terarium "Terarium") of [Blueberry Academy](/wiki/Blueberry_Academy "Blueberry Academy"). First partner Pokémon such as Bulbasaur are raised by [Pokémon Breeders](/wiki/Pok%C3%A9mon_Breeder_(Trainer_class) "Pokémon Breeder (Trainer class)") to be distributed to new [Trainers](/wiki/Pok%C3%A9mon_Trainer "Pokémon Trainer").[[1]](#cite_note-1) Having been raised by [humans](/wiki/Human "Human") from birth, it is regarded as both a rare and well-behaved Pokémon. It is known to be extremely loyal, even after long-term abandonment.[[2]](#cite_note-2) Bulbasaur has demonstrated a nurturing instinct towards younger, weaker Pokémon, such as using its vines to pick up a crying Pokémon and gently rocking it back and forth through the air while singing a "Bulba-by".[[3]](#cite_note-3) Its bulb will flash blue when it is ready to [evolve](/wiki/Evolution "Evolution"), but if it does not want to, it struggles to resist the transformation. Many Bulbasaur gather every year in a hidden garden in Kanto to evolve into [Ivysaur](/wiki/Ivysaur_(Pok%C3%A9mon) "Ivysaur (Pokémon)") in a ceremony led by a [Venusaur](/wiki/Venusaur_(Pok%C3%A9mon) "Venusaur (Pokémon)").[[4]](#cite_note-4) Its vines are long and strong enough to allow Bulbasaur to grab the branches of trees and pull itself up to reach [Berries](/wiki/Berry "Berry").[[5]](#cite_note-5) As mentioned in [Pokémon Sleep](/wiki/Pok%C3%A9mon_Sleep "Pokémon Sleep"), Bulbasaur sits still and basks in sunny spots during its sleep, possibly trying to absorb nutrients from its seed to grow big.[[6]](#cite_note-6)  


### Evolution

Bulbasaur evolves into [Ivysaur](/wiki/Ivysaur_(Pok%C3%A9mon) "Ivysaur (Pokémon)"), which evolves into [Venusaur](/wiki/Venusaur_(Pok%C3%A9mon) "Venusaur (Pokémon)").

(For specifics on this Pokémon's Evolution in the games, refer to [Game data→Evolution data](#Evolution_data).)

- | [/wiki/File:0001Bulbasaur.png](/wiki/File:0001Bulbasaur.png) |
| --- |
| Unevolved |
| Bulbasaur [Grass](/wiki/Grass_(type) "Grass (type)")[Poison](/wiki/Poison_(type) "Poison (type)") | → | [/wiki/File:0002Ivysaur.png](/wiki/File:0002Ivysaur.png) |
| --- |
| First Evolution |
| [Ivysaur](/wiki/Ivysaur_(Pok%C3%A9mon) "Ivysaur (Pokémon)") [Grass](/wiki/Grass_(type) "Grass (type)")[Poison](/wiki/Poison_(type) "Poison (type)") | → | [/wiki/File:0003Venusaur.png](/wiki/File:0003Venusaur.png) |
| --- |
| Second Evolution |
| [Venusaur](/wiki/Venusaur_(Pok%C3%A9mon) "Venusaur (Pokémon)") [Grass](/wiki/Grass_(type) "Grass (type)")[Poison](/wiki/Poison_(type) "Poison (type)") |

## Game data

### Pokédex entries

| | Generation I |  |  | [Kanto](/wiki/List_of_Pok%C3%A9mon_by_Kanto_Pok%C3%A9dex_number "List of Pokémon by Kanto Pokédex number")    #001 | | --- | --- | --- | --- | | | [Red(JPN)](/wiki/Pok%C3%A9mon_Red_and_Green "Pokémon Red and Green") | *(This entry was originally untranslated in English until it was [reused](/wiki/Pok%C3%A9dex_entry_recycling "Pokédex entry recycling") in [Pokémon FireRed](/wiki/Pok%C3%A9mon_FireRed_and_LeafGreen_Versions "Pokémon FireRed and LeafGreen Versions").)* | | --- | --- | | [Green](/wiki/Pok%C3%A9mon_Red_and_Green "Pokémon Red and Green") |  | | [Red(ENG)](/wiki/Pok%C3%A9mon_Red_and_Blue "Pokémon Red and Blue") | A strange seed was planted on its back at birth. The plant sprouts and grows with this Pokémon. | | [Blue](/wiki/Pok%C3%A9mon_Red_and_Blue "Pokémon Red and Blue") |  | | [Yellow](/wiki/Pok%C3%A9mon_Yellow "Pokémon Yellow") | It can go for days without eating a single morsel. In the bulb on its back, it stores energy. | | [Stadium](/wiki/Pok%C3%A9mon_Stadium "Pokémon Stadium") | The bulb-like pouch on its back grows larger as it ages. The pouch is filled with numerous seeds. | | | | | |
| --- |
| | Generation II |  |  | [Johto](/wiki/List_of_Pok%C3%A9mon_by_Johto_Pok%C3%A9dex_number "List of Pokémon by Johto Pokédex number")    #226 | | --- | --- | --- | --- | | | [Gold](/wiki/Pok%C3%A9mon_Gold "Pokémon Gold") | The seed on its back is filled with nutrients. The seed grows steadily larger as its body grows. | | --- | --- | | [Silver](/wiki/Pok%C3%A9mon_Silver "Pokémon Silver") | It carries a seed on its back right from birth. As it grows older, the seed also grows larger. | | [Crystal](/wiki/Pok%C3%A9mon_Crystal "Pokémon Crystal") | While it is young, it uses the nutrients that are stored in the seeds on its back in order to grow. | | [Stadium 2](/wiki/Pok%C3%A9mon_Stadium_2 "Pokémon Stadium 2") | The seed on its back is filled with nutrients. The seed grows steadily larger as its body grows. *(Pokémon Red, Silver, or Crystal inserted)*   It carries a seed on its back right from birth. As it grows older, the seed also grows larger. *(Pokémon Blue, Gold, or Yellow inserted)* | | | | | |
| | Generation III |  | [Hoenn](/wiki/List_of_Pok%C3%A9mon_by_Hoenn_Pok%C3%A9dex_number "List of Pokémon by Hoenn Pokédex number")    #— |  | [Kanto](/wiki/List_of_Pok%C3%A9mon_by_Kanto_Pok%C3%A9dex_number "List of Pokémon by Kanto Pokédex number")    #001 | | --- | --- | --- | --- | --- | | | [Ruby](/wiki/Pok%C3%A9mon_Ruby_and_Sapphire "Pokémon Ruby and Sapphire") | Bulbasaur can be seen napping in bright sunlight. There is a seed on its back. By soaking up the sun's rays, the seed grows progressively larger. | | --- | --- | | [Sapphire](/wiki/Pok%C3%A9mon_Ruby_and_Sapphire "Pokémon Ruby and Sapphire") |  | | [Emerald](/wiki/Pok%C3%A9mon_Emerald "Pokémon Emerald") |  | | [FireRed](/wiki/Pok%C3%A9mon_FireRed "Pokémon FireRed") | There is a plant seed on its back right from the day this Pokémon is born. The seed slowly grows larger. | | [LeafGreen](/wiki/Pok%C3%A9mon_LeafGreen "Pokémon LeafGreen") | A strange seed was planted on its back at birth. The plant sprouts and grows with this Pokémon. | | | | | | |
| | Generation IV |  | [Sinnoh](/wiki/List_of_Pok%C3%A9mon_by_Sinnoh_Pok%C3%A9dex_number "List of Pokémon by Sinnoh Pokédex number")    #— |  | [Johto](/wiki/List_of_Pok%C3%A9mon_by_Johto_Pok%C3%A9dex_number "List of Pokémon by Johto Pokédex number")    #231 | | --- | --- | --- | --- | --- | | | [Diamond](/wiki/Pok%C3%A9mon_Diamond_and_Pearl "Pokémon Diamond and Pearl") | For some time after its birth, it grows by gaining nourishment from the seed on its back. | | --- | --- | | [Pearl](/wiki/Pok%C3%A9mon_Diamond_and_Pearl "Pokémon Diamond and Pearl") |  | | [Platinum](/wiki/Pok%C3%A9mon_Platinum "Pokémon Platinum") |  | | [HeartGold](/wiki/Pok%C3%A9mon_HeartGold "Pokémon HeartGold") | The seed on its back is filled with nutrients. The seed grows steadily larger as its body grows. | | [SoulSilver](/wiki/Pok%C3%A9mon_SoulSilver "Pokémon SoulSilver") | It carries a seed on its back right from birth. As it grows older, the seed also grows larger. | | | | | | |
| | Generation V |  |  | [Unova](/wiki/List_of_Pok%C3%A9mon_by_Unova_Pok%C3%A9dex_number "List of Pokémon by Unova Pokédex number")    #— | | --- | --- | --- | --- | | | [Black](/wiki/Pok%C3%A9mon_Black_and_White "Pokémon Black and White") | For some time after its birth, it grows by gaining nourishment from the seed on its back. | | --- | --- | | [White](/wiki/Pok%C3%A9mon_Black_and_White "Pokémon Black and White") |  | | [Black 2](/wiki/Pok%C3%A9mon_Black_2_and_White_2 "Pokémon Black 2 and White 2") | For some time after its birth, it grows by gaining nourishment from the seed on its back. | | [White 2](/wiki/Pok%C3%A9mon_Black_2_and_White_2 "Pokémon Black 2 and White 2") |  | | | | | |
| | Generation VI |  | [Central Kalos](/wiki/List_of_Pok%C3%A9mon_by_Central_Kalos_Pok%C3%A9dex_number "List of Pokémon by Central Kalos Pokédex number")    #080 |  | [Coastal Kalos](/wiki/List_of_Pok%C3%A9mon_by_Coastal_Kalos_Pok%C3%A9dex_number "List of Pokémon by Coastal Kalos Pokédex number")    #— |  | [Mountain Kalos](/wiki/List_of_Pok%C3%A9mon_by_Mountain_Kalos_Pok%C3%A9dex_number "List of Pokémon by Mountain Kalos Pokédex number")    #— |  | [Hoenn](/wiki/List_of_Pok%C3%A9mon_by_Hoenn_Pok%C3%A9dex_number "List of Pokémon by Hoenn Pokédex number")    #— | | --- | --- | --- | --- | --- | --- | --- | --- | --- | | | [X](/wiki/Pok%C3%A9mon_X "Pokémon X") | A strange seed was planted on its back at birth. The plant sprouts and grows with this Pokémon. | | --- 

*... (truncated) ...*